# Interaction Between Convection and Pulsation — Implementation
# 대류와 맥동의 상호작용 — 구현

**Paper**: Houdek, G. & Dupret, M.-A., *Living Rev. Solar Phys.*, **12**, 8 (2015). DOI: 10.1007/lrsp-2015-8

---

**한국어**: 본 노트북은 Houdek & Dupret (2015)의 주요 개념을 수치적으로 예시한다:
1. 혼합거리 이론(MLT)의 대류 플럭스 `F_c`를 초단열성(superadiabaticity) 함수로 계산
2. 확률적 강제력을 갖는 감쇠 조화 진동자로부터 태양 p-모드 Lorentzian 선 프로파일 유도
3. 태양 외곽층에서 난류 압력 `P_turb/P_gas` 프로파일
4. 모델 외피에 대한 맥동 성장률 η의 진동수 의존성
5. κ-기작: opacity 범프가 고전 맥동성을 구동하는 방식
6. 구동 vs 감쇠 균형 다이어그램

**English**: This notebook numerically illustrates the central concepts of Houdek & Dupret (2015):
1. Mixing-length convective flux `F_c` as a function of superadiabaticity
2. Damped harmonic oscillator with stochastic forcing → solar p-mode line profile
3. Turbulent pressure profile `P_turb/P_gas` in the outer layers
4. Pulsation growth rate η vs frequency for a model envelope
5. κ-mechanism: opacity bump driving classical pulsators
6. Driving–damping balance diagram

In [ ]:
"""Common imports for convection-pulsation demonstrations.

Uses NumPy for arrays, SciPy for stochastic ODE integration, Matplotlib
for visualisation. No external data are required.
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import periodogram

np.random.seed(42)

plt.rcParams.update({
    "figure.figsize": (8.5, 4.5),
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

## 1. Mixing-Length Convective Flux / 혼합거리 대류 플럭스

**한국어**: 표준 Böhm-Vitense MLT에서 대류 열 플럭스는 초단열 기울기 $\beta = \nabla - \nabla_{\rm ad}$ 에 대해 $F_c \propto \beta^{3/2}$ 로 스케일한다. 이는 convection이 초단열성에 극도로 민감함을 보여준다 — 태양 광구 바로 아래에서 β는 ~0.1에서 ~10^-6 까지 급격히 떨어지며, 이에 따라 플럭스 비가 총 플럭스에서 거의 0으로 전이한다.

**English**: In standard Böhm-Vitense MLT the convective heat flux scales as $F_c \propto \beta^{3/2}$ where $\beta = \nabla - \nabla_{\rm ad}$. This strong power-law sensitivity to superadiabaticity means that just below the photosphere, where β drops from ~0.1 to ~10^-6, the convective share of the total flux collapses.

In [ ]:
def mixing_length_flux(beta, rho=1.0, cp=1.0, g=1.0, mixing_length=1.0,
                       delta=1.0, scale_height=1.0):
    """Compute the MLT convective heat flux for given superadiabaticity.

    Args:
        beta: Superadiabaticity (nabla - nabla_ad). Scalar or ndarray.
        rho: Gas density (arbitrary units).
        cp: Specific heat at constant pressure.
        g: Local gravity.
        mixing_length: Mixing length (= alpha * pressure scale height).
        delta: Isobaric expansion coefficient.
        scale_height: Pressure scale height.

    Returns:
        Convective flux F_c in arbitrary but consistent units.
    """
    beta_safe = np.where(beta > 0, beta, 0.0)
    prefactor = 0.5 * rho * cp * np.sqrt(g * delta) * mixing_length**2 \
                / np.sqrt(scale_height)
    return prefactor * beta_safe**1.5


# Sample superadiabaticity profile (schematic — representing outer layers)
beta = np.logspace(-8, 0, 400)
F_c = mixing_length_flux(beta)

fig, ax = plt.subplots()
ax.loglog(beta, F_c, color="C0", lw=2)
ax.set_xlabel(r"Superadiabaticity  $\beta = \nabla - \nabla_{\rm ad}$")
ax.set_ylabel(r"Convective flux  $F_c$  (arb.)")
ax.set_title(r"MLT flux:  $F_c \propto \beta^{3/2}$")
ax.axvline(0.1, color="red", ls=":", label=r"$\beta \sim 0.1$ (photosphere)")
ax.axvline(1e-6, color="gray", ls=":", label=r"$\beta \sim 10^{-6}$ (deep conv.)")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Damped Harmonic Oscillator → Lorentzian Line Profile / 감쇠 조화 진동자 → Lorentzian 선 프로파일

**한국어**: 확률적으로 강제된 감쇠 조화 진동자의 SDE:
$$\ddot{x} + 2\eta\,\dot{x} + \omega_0^2\,x = f(t)$$
여기서 $f(t)$는 가우시안 백색 잡음 (대류에 의한 난류 구동). 파워 스펙트럼은 Lorentzian 모양:
$$P(\omega) = \frac{|F(\omega)|^2}{(\omega^2 - \omega_0^2)^2 + 4\eta^2\omega^2}$$
이는 피크 주변에서 폭 $\Gamma = 2\eta/(2\pi)$ 의 Lorentzian. 태양에서 $\eta \sim 1.5\,\mu$Hz → $\Gamma \sim 1\,\mu$Hz.

**English**: A stochastically driven damped harmonic oscillator satisfies $\ddot{x} + 2\eta\dot{x} + \omega_0^2 x = f(t)$. The power spectrum is Lorentzian with FWHM $\Gamma = 2\eta/(2\pi)$. In the Sun, $\eta \sim 1.5\,\mu$Hz gives $\Gamma \sim 1\,\mu$Hz — the canonical solar p-mode linewidth.

In [ ]:
def simulate_stochastic_oscillator(f0=3.0e-3, gamma_cyclic=1.0e-6,
                                   total_time=5.0e5, dt=10.0):
    """Integrate a stochastically forced damped harmonic oscillator.

    Euler-Maruyama scheme for d^2 x / dt^2 + 2 eta x_dot + omega_0^2 x = noise.

    Args:
        f0: Mode cyclic frequency in Hz (3 mHz for solar p-mode).
        gamma_cyclic: Full-width at half-maximum in Hz (linewidth).
        total_time: Total integration time in seconds.
        dt: Time step in seconds.

    Returns:
        Tuple (time_array, displacement_array).
    """
    omega0 = 2.0 * np.pi * f0
    eta = np.pi * gamma_cyclic  # eta in angular frequency units
    n_steps = int(total_time / dt)
    time = np.arange(n_steps) * dt
    x = np.zeros(n_steps)
    v = np.zeros(n_steps)
    sigma_noise = 1.0
    for i in range(n_steps - 1):
        noise = sigma_noise * np.random.randn() / np.sqrt(dt)
        v[i + 1] = v[i] + dt * (-2.0 * eta * v[i] - omega0**2 * x[i] + noise)
        x[i + 1] = x[i] + dt * v[i + 1]
    return time, x


time, x = simulate_stochastic_oscillator()
freq, psd = periodogram(x, fs=1.0 / (time[1] - time[0]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(time[:5000] / 3600, x[:5000], color="C1", lw=0.6)
ax1.set_xlabel("Time / hr")
ax1.set_ylabel("Displacement (arb.)")
ax1.set_title("Stochastically forced oscillator — time series")

mask = (freq > 1e-3) & (freq < 5e-3)
ax2.semilogy(freq[mask] * 1e3, psd[mask], color="C2")
ax2.axvline(3.0, color="k", ls=":", label=r"$\nu_0 = 3$ mHz")
ax2.set_xlabel(r"Frequency  $\nu$  (mHz)")
ax2.set_ylabel("PSD (arb.)")
ax2.set_title(r"Lorentzian p-mode peak, $\Gamma = 1\,\mu$Hz")
ax2.legend()
plt.tight_layout()
plt.show()

## 3. Turbulent Pressure Profile / 난류 압력 프로파일

**한국어**: 3D 시뮬레이션과 비국소 MLT가 태양 광구 바로 아래에서 $P_{\rm turb}/P_{\rm gas}$ 가 ~0.15 (15%)에 이르는 예리한 피크를 보여주며, 더 깊이 들어가면 감소한다 (Figure 3). δ Sct 별에서는 70%까지 달할 수 있다.

**English**: 3D simulations and nonlocal MLT both show a sharp peak of $P_{\rm turb}/P_{\rm gas} \sim 0.15$ (~15%) just below the solar photosphere, decreasing inward (Figure 3). In δ Sct stars it can reach 70%.

In [ ]:
def turbulent_pressure_profile(depth_mm, peak_depth=0.15, peak_value=0.16,
                               inner_decay=1.5, outer_decay=0.3):
    """Schematic turbulent pressure profile P_turb / P_total vs depth.

    Approximates Figure 3 of Houdek & Dupret (2015): asymmetric peak just
    below the solar photosphere.

    Args:
        depth_mm: Depth below photosphere in Mm (negative = above).
        peak_depth: Depth of maximum in Mm.
        peak_value: Peak amplitude.
        inner_decay: Inward decay scale in Mm.
        outer_decay: Outward decay scale in Mm (shorter).

    Returns:
        Turbulent pressure fraction, broadcast to `depth_mm`.
    """
    depth = np.asarray(depth_mm, dtype=float)
    profile = np.zeros_like(depth)
    above = depth < peak_depth
    below = ~above
    profile[above] = peak_value * np.exp(-((peak_depth - depth[above])
                                           / outer_decay)**2)
    profile[below] = peak_value * np.exp(-(depth[below] - peak_depth)
                                         / inner_decay)
    return profile


depth_grid = np.linspace(-0.5, 2.5, 400)
nonlocal_ml = turbulent_pressure_profile(depth_grid)
trampedach_3d = turbulent_pressure_profile(depth_grid, peak_depth=0.25,
                                           peak_value=0.135, outer_decay=0.4,
                                           inner_decay=1.8)
ludwig_3d = turbulent_pressure_profile(depth_grid, peak_depth=0.30,
                                       peak_value=0.12, outer_decay=0.55,
                                       inner_decay=1.6)

fig, ax = plt.subplots()
ax.plot(depth_grid, nonlocal_ml, label="Nonlocal ML-Model", color="k", lw=2)
ax.plot(depth_grid, trampedach_3d, label="Trampedach et al. (1999)",
        color="C3", ls="--")
ax.plot(depth_grid, ludwig_3d, label="H.-G. Ludwig (2005)",
        color="C0", ls="-.")
ax.set_xlabel("Depth $R_\\odot - r$  (Mm)")
ax.set_ylabel(r"$\overline{\rho u_r^2}\,/\,\hat{p}$")
ax.set_title("Turbulent pressure profile (schematic, Fig. 3 analogue)")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Pulsation Growth Rate η vs Frequency / 맥동 성장률의 진동수 의존성

**한국어**: Gough 비국소 TDC로 계산된 태양 모드의 총 감쇠율 η = η_g + η_t (Figure 9)를 개략적으로 재현. ν_max ≈ 3 mHz 근처에서 감쇠율의 함몰("general κ-mechanism")이 관측된다 — 여기서 η_g는 *탈안정화* (negative)하고 η_t는 안정화.

**English**: Reproduce schematically the solar linear damping rate η = η_g + η_t from Gough's nonlocal TDC (Figure 9). Near ν_max ≈ 3 mHz there is a dip ("general κ-mechanism") where η_g becomes *destabilising* and η_t stabilising.

In [ ]:
def damping_rate_model(frequency_mhz, nu_max=3.0, eta_scale=1.5):
    """Schematic solar p-mode damping rate and its components.

    Approximates Figure 9: total eta, gas (radiative/convective) component,
    and turbulent (momentum) component as functions of cyclic frequency.

    Args:
        frequency_mhz: Cyclic frequency in mHz.
        nu_max: Frequency of maximum solar power in mHz.
        eta_scale: Overall scaling of damping rate in microhertz.

    Returns:
        Dict with arrays for total, eta_g, eta_t.
    """
    nu = np.asarray(frequency_mhz, dtype=float)
    eta_t = eta_scale * np.exp(0.55 * (nu - 1.5)) - eta_scale
    eta_t = np.clip(eta_t, 0, None)
    eta_g = 0.8 * (nu - nu_max)**2 - 0.9 * np.exp(-((nu - nu_max)**2) / 0.25)
    eta_g = eta_g - 0.3
    eta_total = eta_g + eta_t
    return {"nu": nu, "total": eta_total, "eta_g": eta_g, "eta_t": eta_t}


nu_grid = np.linspace(0.5, 5.3, 300)
eta = damping_rate_model(nu_grid)

fig, ax = plt.subplots()
ax.plot(eta["nu"], eta["total"], color="k", lw=2, label=r"$\eta$ (total)")
ax.plot(eta["nu"], eta["eta_g"], color="C3", ls="--", label=r"$\eta_g$ (gas)")
ax.plot(eta["nu"], eta["eta_t"], color="C0", ls="-.", label=r"$\eta_t$ (turb.)")
ax.axhline(0, color="gray", lw=0.8)
ax.axvline(3.0, color="gray", ls=":", label=r"$\nu_{\rm max} = 3\,$mHz")
ax.set_xlabel(r"Frequency $\nu$  (mHz)")
ax.set_ylabel(r"Damping rate $\eta$  ($\mu$Hz)")
ax.set_title("Solar p-mode damping rate (schematic Figure 9)")
ax.legend()
plt.tight_layout()
plt.show()

## 5. κ-Mechanism: Opacity Bump / κ-기작: 불투명도 범프

**한국어**: κ-기작은 부분 이온화 영역(특히 He II at T~4×10⁴ K)에서 opacity $\kappa$가 온도에 따라 *증가*할 때 작동한다. 수축 위상에서 T가 상승하면 κ도 증가해 복사가 가둬지고, 나중에 에너지 방출로 구동한다 ($d\ln\kappa/d\ln T > 0$). 고전 불안정대의 Cepheid, RR Lyrae, δ Scuti 모두 이 기작으로 구동된다.

**English**: The κ-mechanism operates where opacity $\kappa$ *increases* with temperature in partial ionisation zones (especially He II at T~4×10⁴ K). During compression T rises, κ rises, radiation is trapped, and subsequent release drives the pulsation. Cepheids, RR Lyrae, and δ Scuti all pulsate via this mechanism.

In [ ]:
def opacity_with_bumps(log_T):
    """Schematic Rosseland opacity with H and He II ionisation bumps.

    Args:
        log_T: log10(T) in Kelvin.

    Returns:
        log10 opacity (arb. units).
    """
    log_T = np.asarray(log_T, dtype=float)
    kramers = -3.5 * (log_T - 4.0)
    h_bump = 0.8 * np.exp(-((log_T - 4.0)**2) / 0.04)
    he_ii_bump = 1.2 * np.exp(-((log_T - 4.6)**2) / 0.03)
    z_bump = 0.7 * np.exp(-((log_T - 5.3)**2) / 0.02)
    return kramers + h_bump + he_ii_bump + z_bump


log_T_grid = np.linspace(3.8, 6.0, 500)
log_kappa = opacity_with_bumps(log_T_grid)
dlogkappa_dlogT = np.gradient(log_kappa, log_T_grid)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8.5, 6), sharex=True)
ax1.plot(log_T_grid, log_kappa, color="C4", lw=2)
ax1.set_ylabel(r"$\log \kappa$")
ax1.set_title("Opacity with ionisation bumps (schematic)")
ax1.axvspan(3.9, 4.1, color="red", alpha=0.12, label="H bump")
ax1.axvspan(4.5, 4.7, color="blue", alpha=0.12, label="He II bump")
ax1.axvspan(5.2, 5.4, color="green", alpha=0.12, label="Z bump")
ax1.legend()

ax2.plot(log_T_grid, dlogkappa_dlogT, color="C2", lw=2)
ax2.axhline(0, color="k", lw=0.8)
ax2.fill_between(log_T_grid, 0, dlogkappa_dlogT,
                 where=(dlogkappa_dlogT > 0), color="C2", alpha=0.25,
                 label=r"$d\ln\kappa/d\ln T > 0$ (driving)")
ax2.set_xlabel(r"$\log T$  (K)")
ax2.set_ylabel(r"$d\ln\kappa / d\ln T$")
ax2.legend()
plt.tight_layout()
plt.show()

## 6. Driving–Damping Balance Diagram / 구동–감쇠 균형 다이어그램

**한국어**: 작업 적분의 depth 누적 형태 (Figure 5, 6)를 개략적으로 재현. 구동 영역(positive W)이 표면층에 집중되고 깊은 대류영역에서 감쇠. 총 적분이 양이면 모드는 불안정 → 관측 가능.

**English**: Schematic reproduction of the accumulated work integral (Figures 5–7). Driving (W>0) concentrates in the surface ionisation zones while deeper convection damps. If the total integral is positive the mode is unstable and observable.

In [ ]:
def work_integral_accumulated(log_T, driving_sites=((4.0, 0.15, 0.3),
                                                    (4.6, 0.20, 0.4)),
                              damping_site=(5.0, -0.6, 0.5)):
    """Schematic accumulated work integral W(log T).

    Combines localised driving bumps and a broad damping region.

    Args:
        log_T: log10(T) array.
        driving_sites: Sequence of (center, amplitude, width) for driving.
        damping_site: Tuple (center, negative_amplitude, width).

    Returns:
        Tuple (dw_dlogT, cumulative W).
    """
    log_T = np.asarray(log_T, dtype=float)
    dw_dlogT = np.zeros_like(log_T)
    for (center, amp, width) in driving_sites:
        dw_dlogT += amp * np.exp(-((log_T - center)**2) / (2 * width**2))
    c, a, w = damping_site
    dw_dlogT += a * np.exp(-((log_T - c)**2) / (2 * w**2))
    cum = np.cumsum(dw_dlogT) * (log_T[1] - log_T[0])
    return dw_dlogT, cum


log_T_grid = np.linspace(3.8, 5.8, 400)
dw_stable, W_stable = work_integral_accumulated(
    log_T_grid,
    driving_sites=((4.0, 0.10, 0.2), (4.6, 0.12, 0.3)),
    damping_site=(4.8, -0.5, 0.35))
dw_unstable, W_unstable = work_integral_accumulated(
    log_T_grid,
    driving_sites=((4.0, 0.18, 0.2), (4.6, 0.25, 0.3)),
    damping_site=(5.0, -0.25, 0.5))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.plot(log_T_grid, dw_stable, color="C0", lw=2, label="Stable mode")
ax1.plot(log_T_grid, dw_unstable, color="C3", lw=2, label="Unstable mode")
ax1.axhline(0, color="k", lw=0.8)
ax1.set_xlabel(r"$\log T$ (K)")
ax1.set_ylabel(r"$dW/d\log T$ (arb.)")
ax1.set_title("Differential work integral")
ax1.legend()

ax2.plot(log_T_grid, W_stable, color="C0", lw=2, label="Stable mode")
ax2.plot(log_T_grid, W_unstable, color="C3", lw=2, label="Unstable mode")
ax2.axhline(0, color="k", lw=0.8)
ax2.set_xlabel(r"$\log T$ (K)")
ax2.set_ylabel(r"Accumulated $W$ (arb.)")
ax2.set_title("Integrated work (Figures 5, 6 analogue)")
ax2.legend()
plt.tight_layout()
plt.show()

## Summary / 요약

**한국어**: 이 노트북은 Houdek & Dupret (2015)의 핵심 개념을 도식적 코드로 구현했다:
1. **MLT 플럭스**의 β^(3/2) 스케일링 — 대류의 초단열성에 대한 민감도.
2. **Lorentzian 선 프로파일** — 확률적 강제력 + 감쇠 진동자에서 자연 발생.
3. **난류 압력 프로파일** — 태양 광구 아래 ~15% 피크.
4. **감쇠율 η(ν)** — ν_max 근처 "general κ-mechanism" 함몰.
5. **κ-기작** — He II 이온화 영역에서 $d\ln\kappa/d\ln T > 0$가 구동 제공.
6. **작업 적분** — 구동 영역과 감쇠 영역의 경쟁으로 모드 안정성 결정.

실제 TDC 구현은 전체 별 구조 코드(ASTEC, CESAM, MESA)와 맥동 코드(ADIPLS, LOSC)에 Gough 비국소 매개변수 (α, a, b) 또는 Grigahcène 모델의 β̂를 통합해야 한다.

**English**: This notebook implements the key concepts of Houdek & Dupret (2015) schematically:
1. **MLT flux** β^(3/2) scaling — convection is highly sensitive to superadiabaticity.
2. **Lorentzian line profile** — emerges naturally from stochastically forced damped oscillator.
3. **Turbulent pressure profile** — ~15% peak below the solar photosphere.
4. **Damping rate η(ν)** — shows the "general κ-mechanism" dip near ν_max.
5. **κ-mechanism** — opacity derivative $d\ln\kappa/d\ln T > 0$ in the He II zone provides driving.
6. **Work integral** — balance between driving and damping regions determines mode stability.

A full TDC implementation requires embedding Gough's nonlocal parameters (α, a, b) or Grigahcène's β̂ into a stellar structure code (ASTEC, CESAM, MESA) coupled with a pulsation code (ADIPLS, LOSC).

### References
- Houdek, G. & Dupret, M.-A., *Living Rev. Solar Phys.*, **12**, 8 (2015).
- Gough, D. O., *Astrophys. J.*, **214**, 196 (1977a).
- Grigahcène, A. et al., *Astron. Astrophys.*, **434**, 1055 (2005).